# Week 8: RAG + simple ensemble (inference-time)

The full course stack uses **Modal**, **Chroma**, and several agents (`FrontierAgent`, `SpecialistAgent`, `EnsembleAgent`, etc.). This notebook stays **small and runnable**:

1. Load product items from the Hub (same source as earlier weeks).
2. Build a **retrieval index** from a sample of training descriptions with `sentence-transformers` (no Chroma required).
3. **RAG pricer:** retrieve similar products, then ask `gpt-4o-mini` with that context.
4. **Direct pricer:** same model, no retrieval.
5. **Blend:** average RAG and direct estimates (a lightweight stand-in for a full ensemble).

**Requires:** `.env` with `OPENAI_API_KEY`. Optional: `HF_TOKEN` for Hub access.

## 1. Setup

In [ ]:
import os
import re
import sys

import numpy as np
from dotenv import load_dotenv
from openai import OpenAI
from sentence_transformers import SentenceTransformer
from tqdm.auto import tqdm

week8_root = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
sys.path.insert(0, week8_root)

from agents.items import Item
from agents.evaluator import evaluate

load_dotenv(override=True)
client = OpenAI()

EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"
CHAT_MODEL = "gpt-4o-mini"

print("Week8 root:", week8_root)

## 2. Load data and build a retrieval corpus

We embed a subset of **train** summaries as the “catalog.” At inference time we pull the nearest neighbors and pass their prices as hints.

In [ ]:
LITE_MODE = True
USERNAME = "ed-donner"
dataset_name = f"{USERNAME}/items_lite" if LITE_MODE else f"{USERNAME}/items_full"

train, val, test = Item.from_hub(dataset_name)

CORPUS_SIZE = 3000  # lower on slow machines
corpus_items = train[:CORPUS_SIZE]
texts = [it.summary or it.title for it in corpus_items]
prices = np.array([float(it.price) for it in corpus_items], dtype=np.float64)

embedder = SentenceTransformer(EMBED_MODEL)
corpus_emb = embedder.encode(texts, show_progress_bar=True, convert_to_numpy=True)
# normalize for cosine similarity via dot product
corpus_emb = corpus_emb / np.linalg.norm(corpus_emb, axis=1, keepdims=True)

print(f"Corpus vectors: {corpus_emb.shape}")

## 3. Helpers: retrieve + parse price

In [ ]:
TOP_K = 5


def retrieve(description: str):
    q = embedder.encode([description], convert_to_numpy=True)
    q = q / np.linalg.norm(q, axis=1, keepdims=True)
    sims = (corpus_emb @ q[0].T).ravel()
    idx = np.argsort(-sims)[:TOP_K]
    docs = [texts[i] for i in idx]
    p = [prices[i] for i in idx]
    return docs, p


def context_block(docs, p):
    lines = []
    for d, price in zip(docs, p):
        lines.append(f"Related product (reference):\n{d}\nReference price: ${price:.2f}\n")
    return "\n".join(lines)


def parse_price(s: str) -> float:
    s = (s or "").replace("$", "").replace(",", "")
    m = re.search(r"[-+]?\d*\.?\d+", s)
    return float(m.group()) if m else 0.0


def ask_llm(user_text: str) -> float:
    r = client.chat.completions.create(
        model=CHAT_MODEL,
        messages=[{"role": "user", "content": user_text}],
        max_tokens=20,
        temperature=0,
    )
    return parse_price(r.choices[0].message.content or "")

## 4. Three pricers for `evaluate`

In [ ]:
def direct_pricer(item: Item) -> str:
    desc = item.summary or item.title
    prompt = (
        "Estimate the price in USD for this product. Reply with only the number, no words.\n\n"
        f"{desc}"
    )
    return str(ask_llm(prompt))


def rag_pricer(item: Item) -> str:
    desc = item.summary or item.title
    docs, p = retrieve(desc)
    prompt = (
        "Estimate the price in USD for the product below. Use the reference listings only as rough hints;\n"
        "they may not match exactly. Reply with only the number, no words.\n\n"
        f"Product:\n{desc}\n\n{context_block(docs, p)}"
    )
    return str(ask_llm(prompt))


def blend_pricer(item: Item) -> str:
    a = float(direct_pricer(item))
    b = float(rag_pricer(item))
    return str((a + b) / 2.0)

## 5. Run evaluation

Start with a small `EVAL_SIZE` to confirm everything works; increase toward 200 to match the default tester.

In [ ]:
EVAL_SIZE = 50

print("--- Direct (no RAG) ---")
evaluate(direct_pricer, test, size=min(EVAL_SIZE, len(test)))

print("--- RAG ---")
evaluate(rag_pricer, test, size=min(EVAL_SIZE, len(test)))

print("--- Blend (50/50) ---")
evaluate(blend_pricer, test, size=min(EVAL_SIZE, len(test)))

## 6. Where this fits in the full week

- **Chroma + `FrontierAgent`:** Same RAG idea, but the course stores embeddings in a persistent Chroma DB (often built from a large Amazon scrape). You can swap this notebook’s in-memory index for that DB once `products_vectorstore` exists.
- **`SpecialistAgent`:** Calls your fine-tuned model on Modal — week 7’s output, deployed.
- **`EnsembleAgent`:** Combines specialist, frontier, and a small neural model with learned weights; the simple average here is a toy analogue.

When you are ready, run `price_is_right.py` / `DealAgentFramework` from the `week8` folder with Modal and API keys configured.